# Chapter 8. Unsupervised Learning Techniques

Most real-world data is unlabeled.

In supervised learning, the training data includes both input features `X` and labels `y`. In unsupervised learning, the training data only contains the input features `X`, so the algorithm must discover useful structure on its own.

This is useful because labeling data can be expensive, slow, and sometimes require human experts.

In this chapter, we study three main unsupervised learning tasks:

- clustering
- anomaly detection
- density estimation

Chapter 7 already covered dimensionality reduction, another important unsupervised learning task.

This chapter starts with clustering algorithms such as k-means and DBSCAN, then moves on to Gaussian mixture models for density estimation, clustering, and anomaly detection.

## Clustering Algorithms: k-means and DBSCAN

Clustering groups similar instances into clusters without using target labels.

It is useful for tasks such as customer segmentation, anomaly detection, semi-supervised learning, and image segmentation.

Different algorithms define clusters differently: k-means looks for groups around centroids, while DBSCAN looks for dense regions.

### k-Means Clustering

The k-means algorithm is a simple and efficient clustering algorithm.

It works especially well when the data is composed of roughly round clusters, or blobs.

The goal of k-means is to find $k$ cluster centers, called centroids, and assign each instance to the closest centroid.

In [6]:
import os

# Avoid a KMeans memory leak warning on Windows with MKL
os.environ["OMP_NUM_THREADS"] = "8"

In [7]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Define five blob centers and their standard deviations
blob_centers = [
    [0.2, 2.3],
    [-1.5, 2.3],
    [-2.8, 1.8],
    [-2.8, 2.8],
    [-2.8, 1.3],
]

blob_std = [0.4, 0.3, 0.1, 0.1, 0.1]

# Generate a synthetic dataset with five blob-shaped clusters
X, y = make_blobs(n_samples=2000, centers=blob_centers,
                  cluster_std=blob_std, random_state=7)

k = 5

# Train k-means to find five clusters
kmeans = KMeans(n_clusters=k, random_state=42)

y_pred = kmeans.fit_predict(X)

After fitting the model, each instance is assigned to one of the five clusters.

In clustering, the predicted label is the index of the cluster assigned by the algorithm. This is different from classification labels, which represent known target classes.

In [18]:
print("Predicted cluster labels:\n", y_pred)
print("\nAre predicted labels stored in kmeans.labels_?")
print(y_pred is kmeans.labels_)

Predicted cluster labels:
 [2 2 4 ... 1 4 2]

Are predicted labels stored in kmeans.labels_?
True


The `labels_` attribute stores the cluster index assigned to each training instance.

These labels are created by the clustering algorithm itself, not provided as target labels during training.

In [21]:
print("Cluster centres:")
print(kmeans.cluster_centers_.round(3))

Cluster centres:
[[-0.067  2.104]
 [-2.793  2.796]
 [-2.802  1.552]
 [-1.475  2.284]
 [ 0.47   2.414]]


The `cluster_centers_` attribute stores the centroids found by k-means.

Each centroid represents the center of one cluster. K-means assigns each instance to the closest centroid.

In [24]:
import numpy as np

X_new = np.array([
    [0, 2],
    [3, 2],
    [-3, 3],
    [-3, 2.5],
])

print("Predicted clusters for new instances:\n", kmeans.predict(X_new))

Predicted clusters for new instances:
 [0 4 1 1]


New instances are assigned to the cluster whose centroid is closest.

This creates decision regions around the centroids. These regions form a Voronoi tessellation, where every point in the feature space belongs to the nearest centroid.

#### Hard Clustering and Soft Clustering

Assigning each instance to a single cluster is called hard clustering.

However, it can also be useful to measure how close each instance is to every cluster. This is called soft clustering.

In the `KMeans` class, the `transform()` method returns the distance from each instance to every centroid.

In [36]:
print("Distances from new instances to each centroid:\n", kmeans.transform(X_new).round(2))

Distances from new instances to each centroid:
 [[0.12 2.9  2.84 1.5  0.63]
 [3.07 5.85 5.82 4.48 2.56]
 [3.07 0.29 1.46 1.69 3.52]
 [2.96 0.36 0.97 1.54 3.47]]


Each row represents one new instance, and each column represents the distance to one centroid.

If there are $k$ clusters, then `transform()` converts each instance into a $k$-dimensional vector of distances.

This can be used as a form of nonlinear dimensionality reduction or as extra features for another model.

#### The k-means algorithm

The k-means algorithm starts by randomly initializing $k$ centroids.

Then it repeats two main steps:

1. Assign each instance to the closest centroid.
2. Update each centroid by computing the mean of the instances assigned to it.

These two steps are repeated until the centroids stop moving.

Although k-means is guaranteed to converge, it may converge to a suboptimal solution depending on the initial centroid positions.

This is why centroid initialization is important.

#### Centroid initialization methods

K-means can converge to different solutions depending on the initial centroid positions.

If the initial centroids are poorly chosen, the algorithm may converge to a suboptimal local optimum.

One way to reduce this risk is to provide good initial centroids manually.

In [45]:
good_init = np.array([
    [-3, 3],
    [-3, 2],
    [-3, 1],
    [-1, 2],
    [0, 2]
])

kmeans = KMeans(n_clusters=5, init=good_init, random_state=42)

kmeans.fit(X)

print("Cluster centers:\n", kmeans.cluster_centers_.round(3))

Cluster centers:
 [[-2.793  2.796]
 [-2.804  1.801]
 [-2.8    1.301]
 [-1.467  2.286]
 [ 0.209  2.256]]


The `init` hyperparameter can be used to specify the initial centroid positions.

When the initial centroids are provided manually, `n_init=1` is used because there is no need to run multiple random initializations.

Another solution is to run k-means multiple times with different random initializations and keep the best result.

The number of initializations is controlled by the `n_init` hyperparameter.

Scikit-Learn selects the model with the lowest inertia.

In [53]:
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)

kmeans.fit(X)

print("Intertia:\n", kmeans.inertia_)

Intertia:
 219.42800073647598


Inertia is the sum of squared distances between each instance and its closest centroid.

A lower inertia means that instances are closer to their assigned centroids.

However, inertia should be interpreted carefully because it naturally decreases as the number of clusters increases.

In [56]:
print("Negative inertia from score():")
print(kmeans.score(X))

Negative inertia from score():
-219.42800073647595


The `score()` method returns the negative inertia.

This is because Scikit-Learn follows the rule that higher scores should be better.

By default, Scikit-Learn uses `init="k-means++"`.

K-means++ chooses initial centroids more carefully by spreading them out across the dataset.

This makes k-means less likely to converge to a poor local optimum compared to purely random initialization.